# Biomedical Entity Extraction with SciSpaCy

This notebook will use SciSpaCy to identify biomedical entities in de-identified clinical notes.

In [5]:
import spacy

nlp = spacy.load("en_core_sci_sm")
print("Loaded SciSpaCy model: en_core_sci_sm")

Loaded SciSpaCy model: en_core_sci_sm


In [6]:
import pandas as pd

notes = pd.read_csv("../data/sample_clinical_notes.csv")
notes.head()

,patient_id,note
0,1,Patient diagnosed with hypertension. Started o...
1,2,Patient with type 2 diabetes mellitus prescrib...
2,3,Continue atorvastatin 20 mg nightly for manage...
3,4,Patient prescribed levothyroxine for hypothyro...
4,5,Started sertraline to help manage symptoms of ...


## Raw biomedical entity extraction

SciSpaCy identifies spans of text that are likely to be biomedical entities. The next step will evaluate and refine these raw results.

In [7]:
for _, row in notes.iterrows():
    doc = nlp(row["note"])

    print(f"\nPatient {row['patient_id']}")
    print(f"Note: {row['note']}")
    print("Entities:")

    for entity in doc.ents:
        print(f"- {entity.text} ({entity.label_})")


Patient 1
Note: Patient diagnosed with hypertension. Started on lisinopril 10 mg daily.
Entities:
- Patient (ENTITY)
- diagnosed (ENTITY)
- hypertension (ENTITY)
- lisinopril (ENTITY)
- daily (ENTITY)

Patient 2
Note: Patient with type 2 diabetes mellitus prescribed metformin 500 mg twice daily.
Entities:
- Patient (ENTITY)
- type 2 diabetes mellitus (ENTITY)
- prescribed (ENTITY)
- metformin (ENTITY)

Patient 3
Note: Continue atorvastatin 20 mg nightly for management of elevated cholesterol.
Entities:
- atorvastatin (ENTITY)
- management (ENTITY)
- elevated (ENTITY)

Patient 4
Note: Patient prescribed levothyroxine for hypothyroidism management.
Entities:
- Patient (ENTITY)
- levothyroxine (ENTITY)
- hypothyroidism management (ENTITY)

Patient 5
Note: Started sertraline to help manage symptoms of depression and anxiety.
Entities:
- Started sertraline (ENTITY)
- manage (ENTITY)
- symptoms (ENTITY)
- depression (ENTITY)
- anxiety (ENTITY)


## Structured entity table

Each extracted span becomes one row, making the unstructured clinical notes easier to analyze and evaluate.

In [8]:
entity_rows = []

for _, row in notes.iterrows():
    doc = nlp(row["note"])

    for entity in doc.ents:
        entity_rows.append({
            "patient_id": row["patient_id"],
            "entity_text": entity.text,
            "entity_label": entity.label_,
            "start_char": entity.start_char,
            "end_char": entity.end_char,
        })

entities_df = pd.DataFrame(entity_rows)
entities_df

,patient_id,entity_text,entity_label,start_char,end_char
0,1,Patient,ENTITY,0,7
1,1,diagnosed,ENTITY,8,17
2,1,hypertension,ENTITY,23,35
3,1,lisinopril,ENTITY,48,58
4,1,daily,ENTITY,65,70
5,2,Patient,ENTITY,0,7
6,2,type 2 diabetes mellitus,ENTITY,13,37
7,2,prescribed,ENTITY,38,48
8,2,metformin,ENTITY,49,58
9,3,atorvastatin,ENTITY,9,21


## Baseline evaluation

- We are defining a small manually labeled reference set of clinically relevant entities

- Mark every raw prediction as an exact match or not

- list missed entities

- calc precision, recall, and F1

In [9]:
from IPython.display import display

expected_entities = {
    1: ["hypertension", "lisinopril"],
    2: ["type 2 diabetes mellitus", "metformin"],
    3: ["atorvastatin", "cholesterol"],
    4: ["levothyroxine", "hypothyroidism"],
    5: ["sertraline", "depression", "anxiety"],
}

reference_df = pd.DataFrame([
    {"patient_id": patient_id, "entity_text": entity}
    for patient_id, entity_list in expected_entities.items()
    for entity in entity_list
])

entities_df["normalized_entity"] = entities_df["entity_text"].str.lower().str.strip()
reference_df["normalized_entity"] = reference_df["entity_text"].str.lower().str.strip()

predicted_keys = set(zip(entities_df["patient_id"], entities_df["normalized_entity"]))
reference_keys = set(zip(reference_df["patient_id"], reference_df["normalized_entity"]))

true_positives = len(predicted_keys & reference_keys)
false_positives = len(predicted_keys - reference_keys)
false_negatives = len(reference_keys - predicted_keys)

precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)
f1_score = 2 * precision * recall / (precision + recall)

entities_df["exact_match"] = list(zip(
    entities_df["patient_id"], entities_df["normalized_entity"]
))
entities_df["exact_match"] = entities_df["exact_match"].isin(reference_keys)

missed_entities_df = reference_df[
    ~reference_df.apply(
        lambda row: (row["patient_id"], row["normalized_entity"]) in predicted_keys, axis=1
    )
]

print(f"True positives: {true_positives}")
print(f"False positives: {false_positives}")
print(f"False negatives: {false_negatives}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 score: {f1_score:.2f}")

display(entities_df[["patient_id", "entity_text", "exact_match"]])
display(missed_entities_df[["patient_id", "entity_text"]])

True positives: 8
False positives: 12
False negatives: 3
Precision: 0.40
Recall: 0.73
F1 score: 0.52


,patient_id,entity_text,exact_match
0,1,Patient,False
1,1,diagnosed,False
2,1,hypertension,True
3,1,lisinopril,True
4,1,daily,False
5,2,Patient,False
6,2,type 2 diabetes mellitus,True
7,2,prescribed,False
8,2,metformin,True
9,3,atorvastatin,True


,patient_id,entity_text
5,3,cholesterol
7,4,hypothyroidism
8,5,sertraline


## Rule-based clinical refinement

- new pipeline that adds a new 'EntityRuler' vocab that corrects high-value missed concepts and assigns meaningful categories
- original pipeline kept for comparison

In [ ]:
refined_nlp = spacy.load("en_core_sci_sm")
clinical_ruler = refined_nlp.add_pipe(
    "entity_ruler", after="ner", config={"overwrite_ents": True}
)

clinical_ruler.add_patterns([
    {"label": "CONDITION", "pattern": "hypertension"},
    {"label": "MEDICATION", "pattern": "lisinopril"},
    {"label": "MEDICATION", "pattern": "atorvastatin"},
    {"label": "LAB_FINDING", "pattern": "cholesterol"},
])

clinical_note = (
    "Patient diagnosed with hypertension. Started on lisinopril 10 mg daily. "
    "Continue atorvastatin for elevated cholesterol."
)

refined_doc = refined_nlp(clinical_note)
clinical_entities = [
    (entity.text, entity.label_)
    for entity in refined_doc.ents
    if entity.label_ in {"CONDITION", "MEDICATION", "LAB_FINDING"}
]

for text, label in clinical_entities:
    print(f"- {text} ({label})")